# adg-M5: telephone and VoIP channels (GPU part)

Settings: **Accelerator = GPU (T4 or P100)**, **Internet = on**, **Input = dataset `awsaf49/asvpoof-2019-dataset`**. Run with *Save Version, Save & Run All*.

Everything starts from the milestone 4 models trained **without** the leading/trailing silence, so that the silence shortcut does not mix with the channel effect.

1. picks an ffmpeg build that has GSM, AMR-NB and Opus encoders, and stops if none has them;
2. checks the ASVspoof 2019 copy; downloads ASVspoof 2021 LA eval (Zenodo, ODC-By) and its keys (asvspoof.org), MD5-checked. If the keys cannot be fetched, the 2021 part is skipped and the notebook says so;
3. LFCC + wav2vec 2.0 features for the 5,900-file eval subset through 10 simulated channels, for a randomly degraded copy of train and dev (augmentation), and for the ASVspoof 2021 subset;
4. scores the milestone 4 LCNN and GMM (trained without silence) on all of it, and trains an LCNN with channel augmentation.

Every step goes through `run()`, which stops the notebook at the first error.

Output in `/kaggle/working`: `w2v2_m5/` (about 4 GB) and `results/` (score files).

In [ ]:
import os, shutil, subprocess, torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')
print('CPU cores:', os.cpu_count(), '| free disk in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

def run(cmd):
    """Run a shell command, stream its output, raise if it fails."""
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

CHANNELS = ['tel8k', 'g711', 'gsm', 'amr12k', 'opus6k', 'opus12k', 'opus24k', 'loss1', 'loss5', 'loss10']
SUBSET = 'results/m4/band_subset_eval.csv'
FEAT = '/tmp/features'
LFCC = f'{FEAT}/lfcc_asvspoof2019_b02'
W2V = '/kaggle/working/w2v2_m5'
OUT = '/kaggle/working/results/m5'
WORKERS = max(1, os.cpu_count() - 1)

In [ ]:
run('rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/audio-deepfake-generalization.git /tmp/repo')
run('pip install -q soundfile imageio-ffmpeg')
os.chdir('/tmp/repo')

In [ ]:
# ffmpeg with the three codec libraries the channels need
import imageio_ffmpeg
NEED = ['libgsm', 'libopencore_amrnb', 'libopus']
def encoders(exe):
    out = subprocess.run([exe, '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
    return {l.split()[1] for l in out.splitlines() if len(l.split()) > 1}
candidates = [c for c in [shutil.which('ffmpeg'), imageio_ffmpeg.get_ffmpeg_exe()] if c]
for c in candidates:
    print(c, '->', [n for n in NEED if n in encoders(c)])
good = [c for c in candidates if all(n in encoders(c) for n in NEED)]
assert good, 'no ffmpeg build here has GSM, AMR-NB and Opus: send the lines above to Claude'
os.environ['ADG_FFMPEG'] = good[0]
print('using', good[0])

In [ ]:
LA = subprocess.check_output('python scripts/verify_asvspoof_copy.py --find /kaggle/input', shell=True, text=True).strip()
print('ASVspoof 2019 LA copy found in', LA)
run(f'python scripts/verify_asvspoof_copy.py --la-root "{LA}"')
r = subprocess.run('python scripts/download_asvspoof2021.py --out /tmp/asv21', shell=True)
if r.returncode not in (0, 2):
    raise RuntimeError(f'ASVspoof 2021 download failed with code {r.returncode}')
ASV21 = r.returncode == 0
print('ASVspoof 2021 part:', 'ON' if ASV21 else 'SKIPPED (keys unavailable)')
COMMON = f'--la-root "{LA}" --asv21-root /tmp/asv21'

In [ ]:
# LFCC (CPU)
run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions train dev --transform trim40')
run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions train dev --transform trim40+randchan')
run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions eval --subset {SUBSET} --transform trim40')
for ch in CHANNELS:
    run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions eval --subset {SUBSET} --transform trim40+{ch}')
if ASV21:
    run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions asv21 --subset results/m5/asv21_subset.csv --transform trim40')

In [ ]:
# wav2vec 2.0 (GPU; audio decoding and ffmpeg in WORKERS processes)
run(f'python scripts/extract_wav2vec2.py {COMMON} --out {W2V} --device cuda --workers {WORKERS} --partitions train dev --transform trim40+randchan')
for ch in CHANNELS:
    run(f'python scripts/extract_wav2vec2.py {COMMON} --out {W2V} --device cuda --workers {WORKERS} --partitions eval --subset {SUBSET} --transform trim40+{ch}')
if ASV21:
    run(f'python scripts/extract_wav2vec2.py {COMMON} --out {W2V} --device cuda --workers {WORKERS} --partitions asv21 --subset results/m5/asv21_subset.csv --transform trim40')

In [ ]:
# test sets for the LFCC systems
parts = ['evalsub_trim40'] + [f'evalsub_trim40-{ch}' for ch in CHANNELS] + (['asv21sub_trim40'] if ASV21 else [])
P = ' '.join(parts)
# milestone 4 LCNN trained without silence (weights in the repo)
run(f'python scripts/score_lcnn.py --model-dir results/m4/lfcc_lcnn_trim40 --feat-dir {LFCC} {COMMON} --parts {P} --out {OUT}/lfcc_lcnn_trim40 --device cuda')
# milestone 4 GMMs trained without silence
os.makedirs(f'{OUT}/lfcc_gmm_trim40', exist_ok=True)
for p in parts:
    run(f'python scripts/score_gmm.py --gmm-dir results/m4/lfcc_gmm --gmm-suffix _trimmed_40db --feat-dir {LFCC}/{p} --out {OUT}/lfcc_gmm_trim40/scores_{p}.csv')

In [ ]:
# LCNN with channel augmentation: clean + randomly degraded copy of train and dev
run(f'python scripts/train_lcnn.py --feat-dir {LFCC} {COMMON} --out {OUT}/lfcc_lcnn_aug --device cuda '
    f'--train-part train_trim40 train_trim40-randchan --dev-part dev_trim40 dev_trim40-randchan --eval-parts {P}')

In [ ]:
if ASV21:
    run(f'cp /tmp/asv21/asv21_labels.csv /kaggle/working/')
    run('cp results/m5/asv21_subset.csv /kaggle/working/results/m5/')  # subset list written by the 2021 script
run('rm -f /kaggle/working/results/m5/lfcc_lcnn_aug/*.npz')
run('du -sh /kaggle/working/*')
run('ls -la /kaggle/working/results/m5/*')